# Introduction to Bayesian Statistics

This notebook covers the foundations of Bayesian inference:
1. **Bayes' theorem** and its components
2. **Prior, likelihood, and posterior**
3. **Conjugate priors** -- the Beta-Binomial model
4. **Visualising** posterior updating

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
print('Setup complete.')

## 1. Bayes' Theorem

$$P(\theta \mid D) = \frac{P(D \mid \theta)\, P(\theta)}{P(D)}$$

| Term | Name | Meaning |
|------|------|---------|
| $P(\theta)$ | Prior | Belief about $\theta$ before seeing data |
| $P(D \mid \theta)$ | Likelihood | How probable the data is given $\theta$ |
| $P(\theta \mid D)$ | Posterior | Updated belief after seeing data |
| $P(D)$ | Evidence | Normalising constant |

In [ ]:
# Simple discrete example: disease testing
# Prior: P(disease) = 0.01
# Test sensitivity: P(+|disease) = 0.95
# Test specificity: P(-|healthy) = 0.90

p_disease = 0.01
p_healthy = 1 - p_disease
p_pos_given_disease = 0.95
p_pos_given_healthy = 0.10  # false positive rate

# P(+) = P(+|D)P(D) + P(+|H)P(H)
p_pos = p_pos_given_disease * p_disease + p_pos_given_healthy * p_healthy

# Posterior: P(D|+)
p_disease_given_pos = (p_pos_given_disease * p_disease) / p_pos

print(f'P(disease | positive test) = {p_disease_given_pos:.4f}')
print(f'Despite 95% sensitivity, only ~{p_disease_given_pos*100:.1f}% chance of disease!')
print('This is the base rate fallacy -- the prior matters enormously.')

## 2. The Beta-Binomial Conjugate Model

For a coin with unknown bias $\theta$:
- **Prior:** $\theta \sim \text{Beta}(\alpha, \beta)$
- **Likelihood:** $k \mid \theta \sim \text{Binomial}(n, \theta)$
- **Posterior:** $\theta \mid k \sim \text{Beta}(\alpha + k, \beta + n - k)$

The Beta distribution is **conjugate** to the Binomial: the posterior is the same family as the prior.

In [ ]:
# Prior parameters
alpha_prior, beta_prior = 2, 2  # slightly informative, symmetric

# Observed data: 7 heads in 10 flips
n_flips, n_heads = 10, 7

# Posterior parameters
alpha_post = alpha_prior + n_heads
beta_post = beta_prior + (n_flips - n_heads)

theta = np.linspace(0, 1, 500)

fig, ax = plt.subplots()
ax.plot(theta, stats.beta.pdf(theta, alpha_prior, beta_prior), 'b-', lw=2, label=f'Prior: Beta({alpha_prior},{beta_prior})')
ax.plot(theta, stats.beta.pdf(theta, alpha_post, beta_post), 'r-', lw=2, label=f'Posterior: Beta({alpha_post},{beta_post})')
ax.axvline(n_heads / n_flips, ls='--', color='gray', label=f'MLE = {n_heads/n_flips:.1f}')
ax.set_xlabel(r'$\theta$ (coin bias)')
ax.set_ylabel('Density')
ax.set_title('Beta-Binomial: Prior vs Posterior')
ax.legend()
plt.show()

post_mean = alpha_post / (alpha_post + beta_post)
print(f'Posterior mean: {post_mean:.4f} (shrunk towards prior from MLE={n_heads/n_flips:.2f})')

## 3. Sequential Posterior Updating

A key advantage of Bayesian inference: we can update our beliefs as new data arrives.
Today's posterior becomes tomorrow's prior.

In [ ]:
# Simulate sequential coin flips from a biased coin (true theta = 0.6)
np.random.seed(42)
true_theta = 0.6
all_flips = np.random.binomial(1, true_theta, size=100)

# Update in batches of 10
batch_size = 10
alpha, beta_ = 1.0, 1.0  # start with uniform prior

theta_grid = np.linspace(0, 1, 300)
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, 100 // batch_size + 1))

ax.plot(theta_grid, stats.beta.pdf(theta_grid, alpha, beta_), color=colors[0], lw=1.5, label='Prior (n=0)')

for i in range(0, 100, batch_size):
    batch = all_flips[i:i+batch_size]
    alpha += batch.sum()
    beta_ += len(batch) - batch.sum()
    ax.plot(theta_grid, stats.beta.pdf(theta_grid, alpha, beta_),
            color=colors[i // batch_size + 1], lw=1.5, label=f'n={i+batch_size}')

ax.axvline(true_theta, ls='--', color='red', lw=2, label=f'True θ = {true_theta}')
ax.set_xlabel(r'$\theta$')
ax.set_ylabel('Density')
ax.set_title('Sequential Posterior Updating')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 4. Credible Intervals

The Bayesian analogue of confidence intervals. A 95% credible interval contains
$\theta$ with 95% posterior probability.

In [ ]:
# Highest Density Interval (HDI) from the final posterior
from scipy.optimize import minimize_scalar

def hdi_beta(alpha, beta_, prob=0.95):
    """Compute the HDI for a Beta distribution."""
    def interval_width(lower):
        upper = stats.beta.ppf(stats.beta.cdf(lower, alpha, beta_) + prob, alpha, beta_)
        if np.isnan(upper):
            return 1e10
        return upper - lower
    result = minimize_scalar(interval_width, bounds=(0, 1 - prob), method='bounded')
    lo = result.x
    hi = stats.beta.ppf(stats.beta.cdf(lo, alpha, beta_) + prob, alpha, beta_)
    return lo, hi

lo, hi = hdi_beta(alpha, beta_)
post_mean = alpha / (alpha + beta_)

print(f'Posterior: Beta({alpha:.0f}, {beta_:.0f})')
print(f'Posterior mean: {post_mean:.4f}')
print(f'95% HDI: [{lo:.4f}, {hi:.4f}]')
print(f'True theta: {true_theta} -- {"inside" if lo <= true_theta <= hi else "outside"} the HDI')

## Key Takeaways

- Bayesian inference combines **prior knowledge** with **observed data** via Bayes' theorem.
- **Conjugate priors** yield closed-form posteriors (e.g., Beta-Binomial).
- Posteriors can be **updated sequentially** -- today's posterior is tomorrow's prior.
- **Credible intervals** have a direct probabilistic interpretation.

**Next:** When closed-form posteriors are not available, we use MCMC sampling.